# PrizePicks NBA Prop Analysis

This notebook analyzes NBA player prop bets from PrizePicks to identify value opportunities.

## 1. Setup and Imports

Import required libraries and configure project paths.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy import stats
from nba_api.stats.endpoints import leaguedashteamstats
from datetime import datetime

pd.set_option('display.max_columns', None)

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.models.xgboost_model import *
from src.models.ngboost_model import *
from src.points_model import PointsPropModel
from src.utils.helper_functions import findOpp
from src.utils.team_info import projectedStartingFive, mainStartingFive, teamStarPlayer, nameDict, questionablePlayers

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Update Daily Lineups

Scrape the latest NBA starting lineups and injury reports from RotoWire.

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'CHA': ['Ryan Kalkbrenner'], 'IND': ['Johnny Furphy', 'T.J. McConnell'], 'BOS': ['Jordan Walsh'], 'DAL': ['Max Christie', 'Klay Thompson', 'Cooper Flagg'], 'UTA': ['Lauri Markkanen'], 'DEN': ['Peyton Watson'], 'OKC': ['Chet Holmgren', 'Aaron Wiggins', 'Alex Caruso'], 'ORL': ['Tristan da Silva'], 'DET': ['Ronald Holland'], 'POR': ['Jerami Grant']}

Out Players:
{'CHA': ['Grant Williams'], 'CLE': ['Evan Mobley', 'Larry Nance', 'Max Strus'], 'IND': ['Aaron Nesmith', 'Ben Sheppard', 'Obi Toppin'], 'BOS': ['Jayson Tatum'], 'DAL': ['Kyrie Irving'], 'NOP': ['Dejounte Murray'], 'UTA': ['Georges Niang', 'Kevin Love'], 'DEN': ['Christian Braun', 'Aaron Gordon'], 'MEM': ['Brandon Clarke', 'Scotty Pippen', 'Zach Edey', 'Vince Williams', 'Ja Morant', 'Javon Small', 'Ty Jerome', 'John Konchar'], 'OKC': ['Isaiah Hartenstein', 'Ousmane Dieng', 'Jaylin Williams', 'Nikola Topic'], 'ORL': ['Moritz Wagner', 'Colin Castleton', 'Franz Wagner'], 'GSW': ['Seth Curry', 'Al Horford'], 'P

## 3. Load PrizePicks Player Props

Load the latest player point props from PrizePicks data.

In [3]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')


us_df = pd.read_csv(us_file)
us_points = us_df[us_df['CATEGORY'] == 'player_points'].copy()

dfs_df = pd.read_csv(dfs_file)
prizepicks_lines = dfs_df[dfs_df['BOOKMAKER'] == 'PrizePicks'].copy()
pp_points = prizepicks_lines[prizepicks_lines['CATEGORY'] == 'player_points'].copy()

print(f"DFS latest pull: {dfs_df['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {us_df['DATA_PULLED_AT'].max()}")
us_df.head()

DFS latest pull: 2025-12-22 11:15:18
US latest pull: 2025-12-22 11:16:18


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,FanDuel,player_points,Ryan Kalkbrenner,Over,7.5,106,2025-12-23,2025-12-22T17:16:14Z,2025-12-22 11:16:18
1,FanDuel,player_points,Ryan Kalkbrenner,Under,7.5,-140,2025-12-23,2025-12-22T17:16:14Z,2025-12-22 11:16:18
2,FanDuel,player_points,Sion James,Over,5.5,102,2025-12-23,2025-12-22T17:16:14Z,2025-12-22 11:16:18
3,FanDuel,player_points,Sion James,Under,5.5,-136,2025-12-23,2025-12-22T17:16:14Z,2025-12-22 11:16:18
4,FanDuel,player_points,Jaylon Tyson,Over,13.5,-114,2025-12-23,2025-12-22T17:16:14Z,2025-12-22 11:16:18


In [4]:
def american_to_implied(odds):
    """Convert American odds to implied probability."""
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def get_best_us_odds(player_name, line, side, us_points_df):
    """
    Find the best US sportsbook odds that match the PrizePicks line.
    Returns (best_odds, best_book) or (-137, None) if no match found.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return -137, None  # Default to -137 if no match
    
    # Find best odds (highest = least negative or most positive)
    best_idx = player_lines['ODDS'].idxmax()
    best_odds = int(player_lines.loc[best_idx, 'ODDS'])
    best_book = player_lines.loc[best_idx, 'BOOKMAKER']
    
    return best_odds, best_book

def get_market_fair_prob(player_name, line, side, us_points_df):
    """
    Get fair probability from US sportsbook consensus.
    Averages implied probabilities across books and deducts vig.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return None
    
    # Average implied probability across all books
    implied_probs = player_lines['ODDS'].apply(american_to_implied)
    avg_implied = implied_probs.mean()
    
    # Deduct half the vig (~2.5% for -110/-110)
    fair_prob = avg_implied - 0.025
    
    return max(0.01, min(0.99, fair_prob))

# Build PrizePicks props with best matching US odds
pp_props = []

for player in pp_points['NAME'].unique():
    player_data = pp_points[pp_points['NAME'] == player]
    
    over_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Over')]
    under_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Under')]
    
    if over_line.empty:
        continue
        
    line = over_line.iloc[0]['LINE']
    pp_odds = over_line.iloc[0]['ODDS']  # PrizePicks odds (for reference)
    
    # Find best matching US odds for each side
    best_us_odds_over, best_us_book_over = get_best_us_odds(player, line, 'Over', us_points)
    best_us_odds_under, best_us_book_under = get_best_us_odds(player, line, 'Under', us_points)
    
    # Get market fair probabilities from US books
    fair_over = get_market_fair_prob(player, line, 'Over', us_points)
    fair_under = get_market_fair_prob(player, line, 'Under', us_points)
    
    pp_props.append({
        'player': player,
        'line': line,
        'pp_odds': pp_odds,
        'best_us_odds_over': best_us_odds_over,
        'best_us_book_over': best_us_book_over,
        'best_us_odds_under': best_us_odds_under,
        'best_us_book_under': best_us_book_under,
        'fair_prob_over': fair_over,
        'fair_prob_under': fair_under,
    })

pp_props_df = pd.DataFrame(pp_props)
print(f"\nBuilt {len(pp_props_df)} PrizePicks props with best matching US odds")


Built 100 PrizePicks props with best matching US odds


5. Get 2025-26 NBA Data

In [5]:
s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv')

def parse_minutes(min_str):
    if pd.isna(min_str): return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player_name',
    'FGA': 'fga', 'FG3A': 'fg3a', 'FTA': 'fta',
    'FGM': 'fgm', 'FG3M': 'fg3m', 'FTM': 'ftm',
    'PTS': 'pts', 'PLUS_MINUS': 'margin'
})

name_to_id = s26_prepped.groupby('player_name')['player_id'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Get team stats
league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]
team_stats = league_df.set_index('TEAM_ID')

In [6]:
def normalize_name(name):
    return nameDict.get(name, name)

def parse_minutes(min_str):
    if pd.isna(min_str): 
        return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

def build_player_stats_cache(df):
    player_stats = {}
    
    for player_name in df['PLAYER_NAME'].unique():
        player_df = df[df['PLAYER_NAME'] == player_name]
        if len(player_df) < 5:
            continue
            
        recent = player_df.tail(15)
        
        guard = recent['GUARD'].mean() > 0.5 if 'GUARD' in recent.columns else False
        forward = recent['FORWARD'].mean() > 0.5 if 'FORWARD' in recent.columns else False
        center = recent['CENTER'].mean() > 0.5 if 'CENTER' in recent.columns else False
        
        usg_pct = recent['USG_PCT'].mean() if 'USG_PCT' in recent.columns else 0.20
        
        if 'MIN' in recent.columns:
            mins_vals = recent['MIN'].apply(lambda x: parse_minutes(x) if pd.notna(x) else 0)
            minutes = mins_vals.mean()
        else:
            minutes = 20.0
        
        pts_avg = recent['PTS'].mean() if 'PTS' in recent.columns else 10.0
        
        pts_boost_multiplier = 1.0
        
        if 'PTS_DELTA_STAR_OUT' in recent.columns:
            delta = recent['PTS_DELTA_STAR_OUT'].mean()
            
            if pd.notna(delta) and pts_avg > 0:
                raw_multiplier = 1 + (delta / pts_avg)
                if 0.70 <= raw_multiplier <= 1.40:
                    pts_boost_multiplier = raw_multiplier
        
        games_without_star = 0
        if 'GAMES_WITHOUT_STAR' in recent.columns:
            games_without_star = recent['GAMES_WITHOUT_STAR'].iloc[-1] if len(recent) > 0 else 0
        
        player_stats[player_name] = {
            'guard': guard,
            'forward': forward,
            'center': center,
            'usg_pct': usg_pct,
            'minutes': minutes,
            'pts_avg': pts_avg,
            'pts_boost_star_out': pts_boost_multiplier,
            'games_without_star': games_without_star,
            'team': recent['TEAM_ABBREVIATION'].iloc[-1] if 'TEAM_ABBREVIATION' in recent.columns else 'UNK',
            'sample_size': len(recent)
        }
    
    return player_stats

player_stats_cache = build_player_stats_cache(s26)

def get_player_position(player_name):
    stats = player_stats_cache.get(normalize_name(player_name), {})
    positions = []
    if stats.get('guard', False):
        positions.append('guard')
    if stats.get('forward', False):
        positions.append('forward')
    if stats.get('center', False):
        positions.append('center')
    return positions if positions else ['unknown']

def is_same_position(player1, player2):
    pos1 = set(get_player_position(player1))
    pos2 = set(get_player_position(player2))
    return bool(pos1 & pos2)

def is_ball_handler(player_name):
    positions = get_player_position(player_name)
    usg = get_player_usage_rate(player_name)
    return 'guard' in positions and usg > 0.18

def get_player_usage_rate(player_name):
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('usg_pct', 0.20)

def get_player_minutes(player_name):
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('minutes', 20.0)

def get_player_pts_avg(player_name):
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('pts_avg', 10.0)

def get_historical_boost(player_name):
    stats = player_stats_cache.get(normalize_name(player_name), {})
    boost = stats.get('pts_boost_star_out', 1.0)
    games = stats.get('games_without_star', 0)
    return boost, games

def get_game_spread(team_abbrev, current_date, team_lines_dir='data/raw/team_lines'):
    date_str = current_date.replace('-', '')
    team_lines_path = Path(team_lines_dir)
    pattern = f'NBA_{date_str}_*.json'
    matching_files = list(team_lines_path.glob(pattern))
    
    if not matching_files:
        return None
    
    latest_file = max(matching_files, key=lambda p: p.stat().st_mtime)
    
    try:
        with open(latest_file, 'r') as f:
            games_data = json.load(f)
        
        team_name_map = {
            'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
            'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
            'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
            'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
            'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
            'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
            'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
            'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
            'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
            'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
        }
        
        team_full_name = team_name_map.get(team_abbrev)
        if not team_full_name:
            return None
        
        for game in games_data:
            is_home = game['home_team'] == team_full_name
            is_away = game['away_team'] == team_full_name
            
            if is_home or is_away:
                for bookmaker in game['bookmakers']:
                    for market in bookmaker['markets']:
                        if market['market_key'] == 'spreads':
                            for outcome in market['outcomes']:
                                if outcome['name'] == team_full_name:
                                    return outcome['point']
        return None
    except Exception as e:
        return None

def calculate_blowout_prob_from_spread(spread):
    if spread is None:
        return 0.15
    
    expected_margin = -spread
    margin_std = 12.0
    
    prob_win_blowout = 1 - stats.norm.cdf(20, expected_margin, margin_std)
    prob_loss_blowout = stats.norm.cdf(-20, expected_margin, margin_std)
    blowout_prob = prob_win_blowout + prob_loss_blowout
    
    return max(0.05, min(0.40, blowout_prob))

def get_out_and_questionable_players(team_abbrev):
    out = outPlayers.get(team_abbrev, [])
    questionable = questionablePlayers.get(team_abbrev, [])
    return out, questionable

def get_usage_adjustment(player_name, team_abbrev):
    adjustment = 1.0
    reasons = []
    
    player_name_normalized = normalize_name(player_name)
    
    main_lineup = mainStartingFive.get(team_abbrev, [])
    projected = projectedStartingFive.get(team_abbrev, [])
    team_star = teamStarPlayer.get(team_abbrev)
    
    out_players_raw, questionable_players_raw = get_out_and_questionable_players(team_abbrev)
    
    main_normalized = [normalize_name(p) for p in main_lineup]
    projected_normalized = [normalize_name(p) for p in projected]
    out_normalized = [normalize_name(p) for p in out_players_raw]
    questionable_normalized = [normalize_name(p) for p in questionable_players_raw]
    
    if player_name_normalized in out_normalized:
        return 0.0, "Player is OUT - do not bet"
    
    if player_name_normalized in questionable_normalized:
        adjustment *= 0.75
        reasons.append("Player is QUESTIONABLE: -25%")
    
    confirmed_out = []
    
    for player in out_players_raw:
        if normalize_name(player) not in projected_normalized:
            confirmed_out.append(player)
    
    for player in main_lineup:
        player_norm = normalize_name(player)
        if (player_norm not in projected_normalized and 
            player_norm not in out_normalized and
            player_norm not in questionable_normalized):
            confirmed_out.append(player)
    
    player_in_main = player_name_normalized in main_normalized
    player_in_projected = player_name_normalized in projected_normalized
    
    if not confirmed_out and not questionable_players_raw:
        if player_in_main and not player_in_projected:
            adjustment *= 0.85
            reasons.append("Not starting tonight: -15%")
        elif player_in_projected and not player_in_main:
            adjustment *= 1.08
            reasons.append("Starting tonight (expanded): +8%")
        
        adjustment = min(1.30, max(0.70, adjustment))
        return adjustment, "; ".join(reasons) if reasons else "Full strength"
    
    star_is_out = team_star and normalize_name(team_star) in [normalize_name(p) for p in confirmed_out]
    star_is_questionable = team_star and normalize_name(team_star) in questionable_normalized
    
    historical_boost, games_sample = get_historical_boost(player_name)
    has_reliable_historical = (
        games_sample >= 3 and
        0.85 <= historical_boost <= 1.35 and
        historical_boost != 1.0
    )
    
    if has_reliable_historical and star_is_out:
        sample_weight = min(0.8, games_sample / 10)
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        
        adjustment *= regressed_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star out history ({games_sample}g): {boost_pct:+.0f}% → {(regressed_boost-1)*100:+.0f}% regressed")
    
    elif has_reliable_historical and star_is_questionable:
        sample_weight = min(0.8, games_sample / 10)
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        partial_boost = 1.0 + (regressed_boost - 1.0) * 0.5
        
        adjustment *= partial_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star questionable ({games_sample}g history): {boost_pct:+.0f}% → {(partial_boost-1)*100:+.0f}% (50% weighted)")
    
    else:
        for out_player in confirmed_out:
            if normalize_name(out_player) == player_name_normalized:
                continue
            out_usage = get_player_usage_rate(out_player)

            if is_same_position(player_name, out_player):
                capture_rate = 0.30
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(out_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12
                reason_tag = "starter"
            else:
                capture_rate = 0.05
                reason_tag = "bench"
            
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = out_usage * capture_rate
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.18, boost_pct)
                
                if boost_multiplier > 1.02:
                    adjustment *= boost_multiplier
                    reasons.append(f"{out_player} OUT ({reason_tag}): +{(boost_multiplier-1)*100:.0f}%")
        
        for q_player in questionable_players_raw:
            q_player_norm = normalize_name(q_player)
            
            if q_player_norm == player_name_normalized:
                continue
            
            if q_player_norm in [normalize_name(p) for p in confirmed_out]:
                continue

            q_usage = get_player_usage_rate(q_player)
            
            if is_same_position(player_name, q_player):
                capture_rate = 0.30
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(q_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12
                reason_tag = "starter"
            else:
                capture_rate = 0.05
                reason_tag = "bench"
            
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = q_usage * capture_rate * 0.5
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.09, boost_pct)
                
                if boost_multiplier > 1.01:
                    adjustment *= boost_multiplier
                    reasons.append(f"{q_player} QUESTIONABLE ({reason_tag}): +{(boost_multiplier-1)*100:.0f}% (50% weighted)")
    
    if player_in_main and not player_in_projected:
        adjustment *= 0.85
        reasons.append("Not starting tonight: -15%")
    elif player_in_projected and not player_in_main:
        adjustment *= 1.05
        reasons.append("Expanded role: +5%")
    
    adjustment = min(1.30, max(0.0, adjustment))
    
    return adjustment, "; ".join(reasons) if reasons else "No adjustment"

usage_adjustments = {}
adjusted_players = []
out_player_list = []

for player in pp_props_df['player'].unique():
    team = name_to_team.get(player, 'UNK')
    if team == 'UNK':
        usage_adjustments[player] = (1.0, "Unknown team")
        continue
        
    adj, reason = get_usage_adjustment(player, team)
    usage_adjustments[player] = (adj, reason)
    
    if adj == 0.0:
        out_player_list.append({'player': player, 'team': team, 'reason': reason})
    elif adj != 1.0 and reason not in ["Full strength", "No adjustment"]:
        adjusted_players.append({
            'player': player,
            'team': team,
            'adjustment': adj,
            'reason': reason,
            'position': "/".join(get_player_position(player)),
            'usg_pct': get_player_usage_rate(player),
            'pts_avg': get_player_pts_avg(player)
        })

boosts = [p for p in adjusted_players if p['adjustment'] > 1.0]
reductions = [p for p in adjusted_players if p['adjustment'] < 1.0]

print(f"Usage Adjustments: {len(out_player_list)} OUT | {len(boosts)} Boosted | {len(reductions)} Reduced | {len(pp_props_df['player'].unique()) - len(adjusted_players) - len(out_player_list)} Unchanged")

if out_player_list:
    print(f"\n🚫 OUT ({len(out_player_list)}):")
    for p in out_player_list:
        print(f"  {p['player']} ({p['team']})")

if boosts:
    print(f"\n📈 BOOSTED ({len(boosts)}):")
    for p in sorted(boosts, key=lambda x: x['adjustment'], reverse=True)[:10]:
        print(f"  {p['player']} ({p['team']}) {p['adjustment']:.2f}x - {p['reason']}")

Usage Adjustments: 0 OUT | 80 Boosted | 9 Reduced | 11 Unchanged

📈 BOOSTED (80):
  Darius Garland (CLE) 1.30x - Evan Mobley OUT (starter): +10%; Larry Nance OUT (starter): +9%; Max Strus OUT (starter): +9%; Donovan Mitchell OUT (same pos): +18%; De'Andre Hunter OUT (same pos): +18%
  Kon Knueppel (CHA) 1.30x - Grant Williams OUT (starter): +11%; LaMelo Ball OUT (same pos): +18%; Ryan Kalkbrenner QUESTIONABLE (starter): +3% (50% weighted)
  Jarrett Allen (CLE) 1.30x - Evan Mobley OUT (same pos): +18%; Larry Nance OUT (starter): +12%; Max Strus OUT (starter): +12%; Donovan Mitchell OUT (starter): +18%; De'Andre Hunter OUT (starter): +11%
  Jaylon Tyson (CLE) 1.30x - Evan Mobley OUT (starter): +15%; Larry Nance OUT (starter): +14%; Max Strus OUT (starter): +14%; Donovan Mitchell OUT (same pos): +18%; De'Andre Hunter OUT (same pos): +18%; Expanded role: +5%
  Sam Merrill (CLE) 1.30x - Evan Mobley OUT (bench): +6%; Larry Nance OUT (bench): +6%; Max Strus OUT (bench): +6%; Donovan Mitchell 

In [7]:
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

team_abbrev_to_id = s26_prepped.groupby('TEAM_ABBREVIATION')['TEAM_ID'].first().to_dict()

player_projections = {}

for _, row in pp_props_df.iterrows():
    player = row['player']
    player_id = name_to_id.get(player)
    
    if player_id is None:
        continue
    
    usage_adj, adj_reason = usage_adjustments.get(player, (1.0, "No adjustment"))
    
    if usage_adj == 0.0:
        continue
    
    player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    is_b2b = False
    if not player_games.empty:
        latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
        current_date_dt = pd.to_datetime(current_date)
        days_since_last_game = (current_date_dt - latest_game_date).days
        is_b2b = (days_since_last_game == 1)
    
    player_team_abbrev = name_to_team.get(player)
    spread = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
    blowout_prob = calculate_blowout_prob_from_spread(spread)
    
    opp_team_id = None
    try:
        opp_abbrev, _ = findOpp(player, s26, current_date)
        if opp_abbrev and opp_abbrev in team_abbrev_to_id:
            opp_team_id = int(team_abbrev_to_id[opp_abbrev])
    except:
        pass
        
    try:
        projection = model.project_points(
            player_id=player_id,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob, 
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if projection:
            player_projections[player] = {
                'expected': projection['expected_points'],
                'std': projection['std'],
                'team': name_to_team.get(player, 'UNK'),
                'usage_adj': usage_adj,
                'usage_reason': adj_reason
            }
    except:
        continue

In [8]:

def american_to_implied(odds):
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def evaluate_prop_with_volatility(model, player_id, market_line, market_juice, 
                                   player_logs, **projection_kwargs):
    base_eval = model.evaluate_prop(
        player_id=player_id,
        market_line=market_line,
        market_juice=market_juice,
        **projection_kwargs
    )
    
    if not base_eval:
        return None
    
    if len(player_logs) < 15:
        return base_eval
    
    vol_analysis = model.analyze_volatility(player_id)
    
    if not vol_analysis:
        return base_eval
    
    true_prob = model.volatility_analyzer.calculate_true_probability(
        player_logs,
        line=market_line,
        stat_col='pts',
        use_empirical=True
    )
    
    if true_prob['sample_size'] >= 20:
        enhanced_prob_over = true_prob['empirical']['over']
        enhanced_prob_under = true_prob['empirical']['under']
    else:
        enhanced_prob_over = true_prob['t_dist']['over']
        enhanced_prob_under = true_prob['t_dist']['under']
    
    base_edge_over = base_eval['edge_analysis']['over_edge']
    base_edge_under = base_eval['edge_analysis']['under_edge']
    
    market_implied = american_to_implied(market_juice)
    enhanced_edge_over = enhanced_prob_over - market_implied
    enhanced_edge_under = enhanced_prob_under - (1 - market_implied)
    
    mins_vol = vol_analysis['minutes_volatility']
    recency = vol_analysis['recency_bias']
    
    if mins_vol.get('edge_magnitude', 0) > 0:
        volatility_adj = mins_vol['edge_magnitude'] * 0.2
        enhanced_edge_over += volatility_adj
        enhanced_edge_under += volatility_adj
    
    if recency.get('edge_direction'):
        recency_adj = recency['edge_magnitude']
        if recency['edge_direction'] == 'OVER':
            enhanced_edge_over += recency_adj
        elif recency['edge_direction'] == 'UNDER':
            enhanced_edge_under += recency_adj
    
    dist = vol_analysis.get('distribution', {})
    if dist.get('tail_analysis'):
        tail = dist['tail_analysis']
        if tail.get('fat_upper_tail'):
            enhanced_edge_over += 0.015
        if tail.get('fat_lower_tail'):
            enhanced_edge_under += 0.015
    
    best_edge = max(enhanced_edge_over, enhanced_edge_under)
    if best_edge == enhanced_edge_over:
        recommendation = 'OVER'
        best_prob = enhanced_prob_over
    else:
        recommendation = 'UNDER'
        best_prob = enhanced_prob_under
    
    if recommendation == 'OVER':
        ev = enhanced_edge_over * (1 - market_implied)
    else:
        ev = enhanced_edge_under * market_implied
    
    enhanced_eval = base_eval.copy()
    enhanced_eval['edge_analysis']['prob_over'] = enhanced_prob_over
    enhanced_eval['edge_analysis']['prob_under'] = enhanced_prob_under
    enhanced_eval['edge_analysis']['over_edge'] = enhanced_edge_over
    enhanced_eval['edge_analysis']['under_edge'] = enhanced_edge_under
    enhanced_eval['edge_analysis']['recommendation'] = recommendation
    enhanced_eval['edge_analysis']['expected_value'] = ev
    enhanced_eval['edge_analysis']['confidence'] = best_prob
    
    enhanced_eval['volatility_analysis'] = {
        'distribution_fit': dist.get('best_fit', 'normal'),
        'kurtosis': dist.get('kurtosis', 0),
        'skewness': dist.get('skewness', 0),
        'minutes_volatility_edge': mins_vol.get('edge_magnitude', 0),
        'recency_bias_edge': recency.get('edge_magnitude', 0),
        'edge_vs_normal': true_prob.get('edge_vs_normal', {})
    }
    
    return enhanced_eval

def calculate_hit_rate(player_name, line, side, data_df, windows=[5, 10, 15]):
    player_df = data_df[data_df['PLAYER_NAME'] == player_name].copy()
    
    if len(player_df) == 0:
        return {f'L-{w}': None for w in windows}
    
    player_df = player_df.sort_values('GAME_DATE')
    
    results = {}
    
    for window in windows:
        if len(player_df) < window:
            last_n_games = player_df
            actual_window = len(player_df)
        else:
            last_n_games = player_df.tail(window)
            actual_window = window
        
        if actual_window == 0:
            results[f'L-{window}'] = None
            continue
        
        if side.lower() == 'over':
            hits = (last_n_games['PTS'] > line).sum()
        else:
            hits = (last_n_games['PTS'] < line).sum()
        
        hit_rate_pct = (hits / actual_window) * 100
        results[f'L-{window}'] = round(hit_rate_pct, 1)
    
    return results

single_bets = []

for _, row in pp_props_df.iterrows():
    player = row['player']
    line = row['line']
    pp_odds = row['pp_odds']
    
    player_id = name_to_id.get(player)
    if player_id is None:
        continue
    
    usage_adj, adj_reason = usage_adjustments.get(player, (1.0, "No adjustment"))
    
    if usage_adj == 0.0:
        continue
    
    player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    is_b2b = False
    if not player_games.empty:
        latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
        current_date_dt = pd.to_datetime(current_date)
        days_since_last_game = (current_date_dt - latest_game_date).days
        is_b2b = (days_since_last_game == 1)
    
    player_team_abbrev = name_to_team.get(player)
    spread = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
    blowout_prob = calculate_blowout_prob_from_spread(spread)
    
    opp_team_id = None
    try:
        opp_abbrev, _ = findOpp(player, s26, current_date)
        if opp_abbrev and opp_abbrev in team_abbrev_to_id:
            opp_team_id = int(team_abbrev_to_id[opp_abbrev])
    except:
        pass
    
    best_us_odds_over = row.get('best_us_odds_over', pp_odds)
    best_us_odds_under = row.get('best_us_odds_under', pp_odds)
    best_us_book_over = row.get('best_us_book_over', 'PrizePicks')
    best_us_book_under = row.get('best_us_book_under', 'PrizePicks')
    
    fair_over = row.get('fair_prob_over', 0.5) or 0.5
    fair_under = row.get('fair_prob_under', 0.5) or 0.5
    
    pp_implied = american_to_implied(pp_odds)
    
    player_logs = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    
    evaluations = {}
    
    try:
        eval_over = evaluate_prop_with_volatility(
            model=model,
            player_id=player_id,
            market_line=line,
            market_juice=int(best_us_odds_over),
            player_logs=player_logs,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob,
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if eval_over:
            evaluations['over'] = eval_over
    except:
        pass
    
    try:
        eval_under = evaluate_prop_with_volatility(
            model=model,
            player_id=player_id,
            market_line=line,
            market_juice=int(best_us_odds_under),
            player_logs=player_logs,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob,
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if eval_under:
            evaluations['under'] = eval_under
    except:
        pass
    
    if not evaluations:
        continue
    
    edge_over = evaluations['over']['edge_analysis']['over_edge'] if 'over' in evaluations else -999
    edge_under = evaluations['under']['edge_analysis']['under_edge'] if 'under' in evaluations else -999
    
    if 'over' in evaluations:
        model_prob_over = evaluations['over']['edge_analysis']['prob_over']
        edge_vs_pp_over = model_prob_over - 0.50
    else:
        model_prob_over = 0.5
        edge_vs_pp_over = 0.0
    
    if 'under' in evaluations:
        model_prob_under = evaluations['under']['edge_analysis']['prob_under']
        edge_vs_pp_under = model_prob_under - 0.50
    else:
        model_prob_under = 0.5
        edge_vs_pp_under = 0.0
    
    if 'over' in evaluations:
        projection_value = evaluations['over']['projection']['expected_points']
    elif 'under' in evaluations:
        projection_value = evaluations['under']['projection']['expected_points']
    else:
        continue
    
    if projection_value > line:
        best_side = 'Over'
        if 'over' not in evaluations:
            continue
        best_eval = evaluations['over']
        best_model_prob = model_prob_over
        best_fair_prob = fair_over
        best_edge = edge_over
        best_edge_vs_pp = edge_vs_pp_over
        best_us_odds = best_us_odds_over
        best_us_book = best_us_book_over
    else:
        best_side = 'Under'
        if 'under' not in evaluations:
            continue
        best_eval = evaluations['under']
        best_model_prob = model_prob_under
        best_fair_prob = fair_under
        best_edge = edge_under
        best_edge_vs_pp = edge_vs_pp_under
        best_us_odds = best_us_odds_under
        best_us_book = best_us_book_under
    
    if pd.isna(best_us_odds) or best_us_odds is None:
        best_us_odds = pp_odds
        best_us_book = f'Default ({pp_odds})'
    
    projection = best_eval['projection']
    edge_analysis = best_eval['edge_analysis']
    
    base_projection = projection['expected_points']
    
    if best_side == 'Under':
        display_projection = base_projection
        if base_projection > line:
            pts_diff_for_confidence = abs(base_projection - line)
        else:
            pts_diff_for_confidence = abs(line - base_projection)
    else:
        display_projection = base_projection
        pts_diff_for_confidence = abs(base_projection - line)
    
    if best_side == 'Over':
        odds_decimal = (100 / abs(best_us_odds)) + 1 if best_us_odds < 0 else (best_us_odds / 100) + 1
        ev = (best_model_prob * (odds_decimal - 1)) - ((1 - best_model_prob) * 1)
        
        if best_edge > 0 and best_us_odds != 0:
            b = odds_decimal - 1
            kelly_full = (b * best_model_prob - (1 - best_model_prob)) / b
            kelly_full = max(0, kelly_full)
        else:
            kelly_full = 0
    else:
        odds_decimal = (100 / abs(best_us_odds)) + 1 if best_us_odds < 0 else (best_us_odds / 100) + 1
        ev = (best_model_prob * (odds_decimal - 1)) - ((1 - best_model_prob) * 1)
        
        if best_edge > 0 and best_us_odds != 0:
            b = odds_decimal - 1
            kelly_full = (b * best_model_prob - (1 - best_model_prob)) / b
            kelly_full = max(0, kelly_full)
        else:
            kelly_full = 0
    
    kelly_quarter = kelly_full * 0.25
    
    confidence = 'HIGH' if pts_diff_for_confidence > 1.5 * projection['std'] else 'MEDIUM' if pts_diff_for_confidence > projection['std'] else 'LOW'
    
    hit_rates = calculate_hit_rate(player, line, best_side, s26)
    
    single_bets.append({
        'player': player,
        'team': name_to_team.get(player, 'UNK'),
        'line': line,
        'side': best_side,
        'projection': round(display_projection, 1),
        'std': round(projection['std'], 1),
        'model_prob': round(best_model_prob, 3),
        'fair_prob': round(best_fair_prob, 3) if best_fair_prob else None,
        'pp_odds': pp_odds,
        'pp_implied': round(pp_implied, 3),
        'best_us_odds': int(best_us_odds) if not pd.isna(best_us_odds) else pp_odds,
        'best_us_book': best_us_book if best_us_book else f'Default ({pp_odds})',
        'edge_vs_fair': round(best_edge, 4) if best_fair_prob else None,
        'edge_vs_pp': round(best_edge_vs_pp, 4),
        'ev': round(ev, 4),
        'ev_percent': round(ev * 100, 2),
        'kelly_quarter': round(kelly_quarter, 4),
        'confidence': confidence,
        'edge_over': round(edge_analysis['over_edge'], 4),
        'edge_under': round(edge_analysis['under_edge'], 4),
        'usage_adj': usage_adj,
        'usage_reason': adj_reason,
        'L-5': hit_rates.get('L-5'),
        'L-10': hit_rates.get('L-10'),
        'L-15': hit_rates.get('L-15'),
    })

singles_df = pd.DataFrame(single_bets)
singles_df = singles_df.sort_values('edge_vs_fair', ascending=False, na_position='last')

print("\nTOP PRIZEPICKS SINGLE PICKS (by edge vs market)")
print(f"{'Player':<25} {'Tm':<4} {'Side':<5} {'Line':<5} {'Proj':<6} {'Model%':<7} {'Fair%':<7} {'Edge%':<7} {'US Odds':<16} {'EV%':<7} {'Kelly':<7} {'Conf':<5}")
print("-" * 120)

for _, bet in singles_df.head(20).iterrows():
    fair_str = f"{bet['fair_prob']*100:.1f}%" if bet['fair_prob'] else "N/A"
    edge_str = f"{bet['edge_vs_fair']*100:.1f}%" if bet['edge_vs_fair'] else f"{bet['edge_vs_pp']*100:.1f}%*"
    us_odds_str = f"{bet['best_us_odds']:+d}" if bet['best_us_odds'] >= 0 else f"{bet['best_us_odds']}"
    
    print(f"{bet['player']:<25} {bet['team']:<4} {bet['side']:<5} {bet['line']:<5.1f} {bet['projection']:<6.1f} "
          f"{bet['model_prob']*100:<7.1f} {fair_str:<7} {edge_str:<7} {us_odds_str:<6} ({bet['best_us_book']:<8}) "
          f"{bet['ev_percent']:<7.1f} {bet['kelly_quarter']*100:<7.2f} {bet['confidence']:<5}")

adjusted_bets = singles_df[singles_df['usage_adj'] != 1.0]
if len(adjusted_bets) > 0:
    print(f"\nUSAGE-ADJUSTED PLAYERS ({len(adjusted_bets)} total):")
    for _, bet in adjusted_bets.head(10).iterrows():
        direction = "↑" if bet['usage_adj'] > 1 else "↓"
        print(f"{direction} {bet['player']} ({bet['team']}): {bet['usage_adj']:.2f}x | {bet['side']} {bet['line']} (Proj: {bet['projection']}, Edge: {bet['edge_vs_pp']*100:.1f}%) | {bet['usage_reason']}")


TOP PRIZEPICKS SINGLE PICKS (by edge vs market)
Player                    Tm   Side  Line  Proj   Model%  Fair%   Edge%   US Odds          EV%     Kelly   Conf 
------------------------------------------------------------------------------------------------------------------------
Sam Merrill               CLE  Over  7.5   16.3   99.2    52.4%   45.7%   -115   (Bovada  ) 85.4    24.56   HIGH 
Tyus Jones                ORL  Under 5.0   3.0    82.1    nan%    40.2%   -137   (Default (-137)) 42.1    14.42   LOW  
De'Anthony Melton         GSW  Over  6.0   9.5    95.5    nan%    37.7%   -137   (Default (-137)) 65.3    22.36   HIGH 
Zion Williamson           NOP  Over  19.5  26.4   90.9    53.1%   36.3%   -120   (FanDuel ) 66.6    19.99   MEDIUM
Dean Wade                 CLE  Under 6.0   6.0    74.1    nan%    32.3%   -137   (Default (-137)) 28.1    9.64    LOW  
Paolo Banchero            ORL  Under 25.0  24.7   71.2    nan%    31.1%   -137   (Default (-137)) 23.2    7.95    LOW  
Anthony 

In [9]:
# =============================================================================
# 6. SAVE RESULTS
# =============================================================================

# Prepare output DataFrame
output_df = singles_df[[
    'player', 'team', 'line', 'side', 'projection', 'std',
    'model_prob', 'fair_prob', 'pp_odds', 'pp_implied',
    'best_us_odds', 'best_us_book',
    'edge_vs_fair', 'edge_vs_pp', 'ev', 'ev_percent', 'kelly_quarter',
    'usage_adj', 'usage_reason', 'L-5', 'L-10', 'L-15'
]].rename(columns={
    'player': 'NAME',
    'team': 'TEAM',
    'line': 'LINE',
    'side': 'SIDE',
    'projection': 'PREDICTION',
    'std': 'STD',
    'model_prob': 'MODEL_PROB',
    'fair_prob': 'FAIR_PROB',
    'pp_odds': 'PRIZEPICKS_ODDS',
    'pp_implied': 'PRIZEPICKS_IMPLIED',
    'best_us_odds': 'BEST_US_ODDS',
    'best_us_book': 'BEST_US_BOOK',
    'edge_vs_fair': 'EDGE_VS_FAIR',
    'edge_vs_pp': 'EDGE_VS_PP',
    'ev': 'EV',
    'ev_percent': 'EV_PERCENT',
    'kelly_quarter': 'KELLY_QUARTER',
    'usage_adj': 'USAGE_ADJ',
    'usage_reason': 'USAGE_REASON',
    'L-5': 'L5',
    'L-10': 'L10',
    'L-15': 'L15'
})

# Save
from datetime import datetime
today = datetime.now().strftime('%Y-%m-%d')
output_path = f'data/props/ev_analysis/prizepicks.csv'
output_df.to_csv(output_path, index=False)
print(f"\n✓ Saved to {output_path}")
print(f"✓ Total picks: {len(output_df)}")
print(f"✓ Picks with >2% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.02])}")
print(f"✓ Picks with >5% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.05])}")
print(f"✓ Picks using default -137: {len(singles_df[singles_df['best_us_book'] == 'Default (-137)'])}")
print(f"✓ Picks with usage adjustments: {len(singles_df[singles_df['usage_adj'] != 1.0])}")


✓ Saved to data/props/ev_analysis/prizepicks.csv
✓ Total picks: 89
✓ Picks with >2% edge: 43
✓ Picks with >5% edge: 35
✓ Picks using default -137: 28
✓ Picks with usage adjustments: 89


In [10]:
df = singles_df[['player', 'side', 'line', 'projection', 'model_prob','ev_percent', 'kelly_quarter', 'L-5', 'L-10', 'L-15']].sort_values('ev_percent', ascending=False).head(20)
df

,player,side,line,projection,model_prob,ev_percent,kelly_quarter,L-5,L-10,L-15
10,Sam Merrill,Over,7.5,16.3,0.992,85.42,0.2456,80.0,90.0,91.7
31,Zion Williamson,Over,19.5,26.4,0.909,66.62,0.1999,40.0,60.0,61.5
80,De'Anthony Melton,Over,6.0,9.5,0.955,65.28,0.2236,40.0,50.0,50.0
36,Jeremiah Fears,Over,12.5,18.6,0.759,51.72,0.1293,40.0,70.0,66.7
33,Jordan Poole,Over,15.5,18.8,0.775,47.98,0.1319,60.0,60.0,63.6
56,Santi Aldama,Under,14.5,14.0,0.679,42.50,0.0966,40.0,70.0,66.7
88,Tyus Jones,Under,5.0,3.0,0.821,42.10,0.1442,40.0,40.0,53.3
44,Ace Bailey,Under,13.5,12.1,0.741,41.41,0.1139,60.0,70.0,66.7
67,Shaedon Sharpe,Under,24.5,24.4,0.708,38.29,0.1005,60.0,70.0,53.3
3,Brandon Miller,Over,19.5,22.3,0.699,37.13,0.0965,40.0,40.0,41.7
